# ADS Homework 3 - Part 4: Transformer (PyTorch)


This notebook follows `.cursor/rules/task_description_hw3.mdc` and the plan in `hw3_plan.md`.


**Dataset (Kaggle path):**
- **Jena Climate**: `/kaggle/input/jena-climate`

We use PyTorch Transformer encoder layers for time-series forecasting and comparison with RNN/LSTM baselines.

This notebook also includes the bonus research report.

**Author:** [Your Name]


In [ ]:
# Core imports and setup
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


## 1) Dataset Path
Confirm this path matches the Kaggle mount.

- **Jena Climate**: `/kaggle/input/jena-climate`


In [ ]:
# Update if your Kaggle path differs
JENA_ROOT = Path("/kaggle/input/jena-climate")

print("Jena Climate exists:", JENA_ROOT.exists())


# Part 4: Transformer on Jena Climate (Time Series)

**Task:** Forecast `T (degC)` (next step or future window).

**Approach:** Use a Transformer encoder layer (`nn.TransformerEncoder`).

**Comparison:** Compare performance (MAE/MSE, convergence speed) vs RNN/LSTM baselines.


In [ ]:
def load_jena_data(root_path):
    csvs = list(root_path.glob("*.csv"))
    if not csvs:
        return None
    df = pd.read_csv(csvs[0])
    if "T (degC)" in df.columns:
        series = df["T (degC)"].values.astype(np.float32)
    else:
        series = df.iloc[:, 1].values.astype(np.float32)
    return series


def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i + seq_len])
        y.append(data[i + seq_len])
    return np.array(X), np.array(y)


train_data = None
val_data = None

if JENA_ROOT.exists():
    temperature_data = load_jena_data(JENA_ROOT)
    if temperature_data is None:
        print("Jena dataset not found inside the root path.")
    else:
        mean_temp = temperature_data.mean()
        std_temp = temperature_data.std()
        data_norm = (temperature_data - mean_temp) / std_temp

        n = len(data_norm)
        train_data = data_norm[:int(0.7 * n)]
        val_data = data_norm[int(0.7 * n):int(0.9 * n)]
else:
    print("Jena dataset not found.")


def get_ts_loaders(seq_len=24, batch_size=64):
    if train_data is None or val_data is None:
        return None, None

    X_train, y_train = create_sequences(train_data, seq_len)
    X_val, y_val = create_sequences(val_data, seq_len)

    train_ds = TensorDataset(torch.tensor(X_train).unsqueeze(-1), torch.tensor(y_train))
    val_ds = TensorDataset(torch.tensor(X_val).unsqueeze(-1), torch.tensor(y_val))

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False),
    )


In [ ]:
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class LSTMRegressor(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1, dropout=0.0):
        super().__init__()
        self.rnn = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        return self.fc(out).squeeze()


class TransformerTS(nn.Module):
    def __init__(self, input_size=1, d_model=64, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 1)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        out = self.transformer(x)
        out = out[:, -1, :]
        return self.fc(out).squeeze()


def train_ts(model, train_loader, val_loader, epochs=5, lr=1e-3):
    if train_loader is None or val_loader is None:
        return None

    model.to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {"train_mse": [], "val_mse": [], "val_mae": []}

    for epoch in range(1, epochs + 1):
        model.train()
        train_mse = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            train_mse += loss.item()

        train_mse /= max(1, len(train_loader))

        model.eval()
        val_mse = 0.0
        val_mae = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                preds = model(xb)
                val_mse += criterion(preds, yb).item()
                val_mae += nn.L1Loss()(preds, yb).item()

        val_mse /= max(1, len(val_loader))
        val_mae /= max(1, len(val_loader))

        history["train_mse"].append(train_mse)
        history["val_mse"].append(val_mse)
        history["val_mae"].append(val_mae)

        print(
            f"Epoch {epoch} | Train MSE: {train_mse:.4f} | "
            f"Val MSE: {val_mse:.4f} | Val MAE: {val_mae:.4f}"
        )

    return history


def plot_ts_curves(history, title):
    if history is None:
        return
    plt.figure(figsize=(8, 4))
    plt.plot(history["train_mse"], label="train_mse")
    plt.plot(history["val_mse"], label="val_mse")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.legend()
    plt.show()


In [ ]:
if JENA_ROOT.exists() and train_data is not None:
    PLOT_TS_CURVES = True

    train_loader, val_loader = get_ts_loaders(seq_len=24, batch_size=64)

    print("
[LSTM baseline] seq_len=24")
    lstm_model = LSTMRegressor(hidden_size=64, num_layers=1, dropout=0.0)
    hist_lstm = train_ts(lstm_model, train_loader, val_loader, epochs=3, lr=1e-3)

    print("
[Transformer] seq_len=24")
    transformer_model = TransformerTS(d_model=64, nhead=4, num_layers=2, dropout=0.1)
    hist_tf = train_ts(transformer_model, train_loader, val_loader, epochs=3, lr=1e-3)

    if PLOT_TS_CURVES:
        plot_ts_curves(hist_lstm, "LSTM baseline")
        plot_ts_curves(hist_tf, "Transformer")


### Discussion Question (Transformer)
* **What are the main advantages and disadvantages of Transformer-based models?**
* **Why do Transformers scale well with data and model size?**
* **Why do they often require large computational resources compared to simpler models?**
* **What is self-attention, and what problem does it solve?**
* **Why can attention model long-range dependencies more effectively than simple RNNs?**
* **What is multi-head attention, and why does it help?**
* **What is the role of positional encoding?**

*(Double-click to edit)*


# Research (Bonus)
## Which Machine Learning Models Are Actually Used in Industry?

**Part 1: Current usage**
- Summarize which model families are most widely used today.
- Cite credible sources (Kaggle reports, industry surveys, company blogs).

**Part 2: 5-10 year prediction**
- 2-3 paragraphs forecasting shifts in model usage.
- Discuss classical models vs deep learning/LLMs and domain-specific changes.

*(Write your 1-3 page report here with citations.)*


## Wrap-up
- Summarize key findings.
- Optional: short error analysis.
